# Классификатор изображений кошек и собак (ConvNet с нуля)

Ноутбук подготовлен на основе прототипа `code/classify_cats_and_dogs.py` для запуска в **Google Colab**.

**Перед запуском включите GPU:** *Среда выполнения → Сменить среду выполнения → T4 GPU*.

Данные: архив `my_data.tar.gz` в корне Google Drive (структура `cats_and_dogs/{train,validation,test}/{Cats,Dogs}`, см. README.md).

Порядок работы:

1. Проверка окружения
2. Подключение Google Drive
3. Распаковка данных в `/content`
4. Датасеты
5. Проверка целостности изображений
6. Модель
7. Обучение
8. Графики
9. Оценка на контрольном наборе
10. Сохранение модели на Google Drive

## 1. Проверка окружения

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU:", gpus)
else:
    print("GPU не обнаружен — включите: Среда выполнения → Сменить среду выполнения → GPU")

## 2. Подключение Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 3. Распаковка данных

Архив распаковывается на локальный диск виртуальной машины — чтение при обучении идёт с быстрого диска, а не с Drive. Подойдёт любой из двух вариантов.

**Вариант A — ячейка ниже (автоматически):** распаковка через `tarfile` в Python.

**Вариант B — терминал (вручную):**

1. Откройте терминал: панель файлов слева (значок папки) → значок терминала `>_` внизу панели. Терминал стартует в `/content`, поэтому `cd` сработает сразу.
2. Выполните команды:

```bash
cd ./drive/MyDrive
tar -xzf my_data.tar.gz -C /content
```

Если распаковка выполнена вручную, автоматическая ячейка ниже просто выведет «Данные уже распакованы» — дублирующей распаковки не будет.

In [ ]:
import pathlib
import tarfile

# Архив с данными на Google Drive (см. README.md: my_data.tar.gz в корне MyDrive)
ARCHIVE_PATH = pathlib.Path("/content/drive/MyDrive/my_data.tar.gz")
DATA_DIR = pathlib.Path("/content/cats_and_dogs")

if DATA_DIR.exists():
    print(f"Данные уже распакованы: {DATA_DIR}")
else:
    if not ARCHIVE_PATH.exists():
        raise FileNotFoundError(
            f"Архив не найден: {ARCHIVE_PATH}\n"
            "Загрузите my_data.tar.gz в корень Google Drive (MyDrive) и перезапустите ячейку."
        )
    print(f"Распаковка {ARCHIVE_PATH} -> /content ...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall("/content")
    print("Готово.")

if not (DATA_DIR / "train").exists():
    raise FileNotFoundError(f"Не найден каталог {DATA_DIR / 'train'} — проверьте содержимое архива.")

# Количество изображений по каждой выборке
for split in ("train", "validation", "test"):
    for cls in ("Cats", "Dogs"):
        folder = DATA_DIR / split / cls
        n = len(list(folder.iterdir())) if folder.exists() else -1
        print(f"{split}/{cls}: {n}")

## 4. Датасеты

Считываем изображения с диска и выполняем их масштабирование. Имена классов формируются по алфавиту: `Cats` → 0, `Dogs` → 1.

In [2]:
import pathlib

from tensorflow.keras.utils import image_dataset_from_directory

new_base_dir = pathlib.Path("/content/cats_and_dogs")

train_dataset = image_dataset_from_directory(
    new_base_dir / "train",
    image_size=(180, 180),
    batch_size=32)
validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation",
    image_size=(180, 180),
    batch_size=32)
test_dataset = image_dataset_from_directory(
    new_base_dir / "test",
    image_size=(180, 180),
    batch_size=32)

print("class_names:", train_dataset.class_names)

Found 1999 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
class_names: ['Cats', 'Dogs']


## 5. Проверка целостности изображений

В исходном датасете Microsoft встречаются битые файлы (обрезанные JPEG и изображения в формате BMP, переименованные в `.jpg`). На первом батче `model.fit()` они вызывают `InvalidArgumentError: Input size should match (header_size + row_size * abs_height) ...`.

Ячейка ниже декодирует каждый файл тем же способом, каким его читает `image_dataset_from_directory` (`tf.io.decode_image`, 3 канала). Нечитаемые файлы перемещаются в карантин `/content/corrupt_images` с сохранением структуры папок — данные не удаляются. Повторный запуск ячейки безопасен: чистый датасет она просто пропускает. После перемещения файлов в карантин перезапустите ячейку «4. Датасеты», чтобы итераторы пересобрали список файлов.

In [5]:
import pathlib
import shutil

import tensorflow as tf

DATA_DIR = pathlib.Path("/content/cats_and_dogs")
QUARANTINE_DIR = pathlib.Path("/content/corrupt_images")

def is_decodable(path):
    """Проверяет, что файл декодируется так же, как при обучении (decode_image, 3 канала)."""
    try:
        raw = tf.io.read_file(str(path))
        tf.io.decode_image(raw, channels=3, expand_animations=False)
        return True
    except Exception:
        return False

print("Сканирование файлов...")
corrupt = [p for p in sorted(DATA_DIR.rglob("*")) if p.is_file() and not is_decodable(p)]

if not corrupt:
    print("Все файлы декодируются успешно.")
else:
    print(f"Найдено нечитаемых файлов: {len(corrupt)} — перемещаю в карантин {QUARANTINE_DIR}")
    for p in corrupt:
        rel_path = p.relative_to(DATA_DIR)
        dest = QUARANTINE_DIR / rel_path
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(p), str(dest))
        print(f"  карантин: {rel_path}")

Сканирование файлов...
Найдено нечитаемых файлов: 2 — перемещаю в карантин /content/corrupt_images
  карантин: validation/Dogs/2317.jpg
  карантин: validation/Dogs/2494.jpg


По факту, проблемы с файлами validation/Dogs/2317.jpg и validation/Dogs/2494.jpg, которые имеют некорректный размер.

## 6. Модель

Архитектура прототипа: пять сверточных слоев, четыре слоя подвыборки, Flatten и выходной слой с сигмоидой.

In [3]:
from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input(shape=(180, 180, 3))

x = layers.Rescaling(1./255)(inputs)
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.Flatten()(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

model.compile(loss="binary_crossentropy",
              optimizer="rmsprop",
              metrics=["accuracy"])

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 178, 178, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 89, 89, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 87, 87, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 43, 43, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 41, 41, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 18, 18, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 9, 9, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 7, 7, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │        12,545 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 991,041 (3.78 MB)

 Trainable params: 991,041 (3.78 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Обучение

Обучение модели с использованием объекта `Dataset`. Лучшая по `val_loss` модель сохраняется в файл `convnet_from_scratch.keras`.

In [4]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="convnet_from_scratch.keras",
        save_best_only=True,
        monitor="val_loss")
]

history = model.fit(
    train_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=callbacks)

Epoch 1/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - accuracy: 0.5047 - loss: 0.6998

InvalidArgumentError: Graph execution error:

Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) INVALID_ARGUMENT:  Input size should match (header_size + row_size * abs_height) but they differ by 2
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]]
	 [[IteratorGetNext/_2]]
  (1) INVALID_ARGUMENT:  Input size should match (header_size + row_size * abs_height) but they differ by 2
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]]
0 successful operations.
0 derived errors ignored. [Op:__inference_multi_step_on_iterator_2195]

## 8. Графики

Формирование графиков изменения потерь и точности в процессе обучения.

In [ ]:
import matplotlib.pyplot as plt

# Проверка, что history вообще есть и содержит нужные ключи (на случай, если валидации не было)
if "val_accuracy" not in history.history or "val_loss" not in history.history:
    print("Внимание: в истории нет данных валидации (val_accuracy / val_loss).")
else:
    accuracy = history.history["accuracy"]
    val_accuracy = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    epochs = range(1, len(accuracy) + 1)

    # График точности
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, accuracy, "bo", label="Точность на этапе обучения")
    plt.plot(epochs, val_accuracy, "b", label="Точность на этапе проверки")
    plt.title("Точность на этапах обучения и проверки")
    plt.xlabel("Эпоха")
    plt.ylabel("Точность")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.3)

    # График потерь
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, loss, "bo", label="Потери на этапе обучения")
    plt.plot(epochs, val_loss, "b", label="Потери на этапе проверки")
    plt.title("Потери на этапах обучения и проверки")
    plt.xlabel("Эпоха")
    plt.ylabel("Потери")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.3)

    plt.show()

## 9. Оценка на контрольном наборе

In [ ]:
test_model = keras.models.load_model("convnet_from_scratch.keras")
test_loss, test_acc = test_model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")

## 10. Сохранение модели на Google Drive

Копия лучшей модели сохраняется на Google Drive, чтобы пережить сброс сессии Colab.

In [ ]:
import pathlib
import shutil

BEST_MODEL = "convnet_from_scratch.keras"
DEST_PATH = pathlib.Path("/content/drive/MyDrive/convnet_from_scratch.keras")

shutil.copy2(BEST_MODEL, DEST_PATH)
print(f"Лучшая модель скопирована на Google Drive: {DEST_PATH}")